# N-BEATS Walmart Store Sales Forecasting

Generic N-BEATS model for weekly Walmart Store-Dept sales, using only past target
history (no static or covariate features). Compared against seasonal-naive and
last-value-naive baselines, using the same dual validation strategy as the other
models in this project (last-39-weeks and calendar-aligned).

## Setup and Imports

In [40]:
from pathlib import Path
import sys
import os
import json
import random
import time
import shutil
import zipfile
from copy import deepcopy

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

import matplotlib.pyplot as plt

In [41]:
pip install dagshub mlflow pandas matplotlib seaborn skops --quiet

Note: you may need to restart the kernel to use updated packages.


In [42]:
KAGGLE_INPUT = Path("/kaggle/input")
repo_root = Path("/kaggle/working/Walmart").resolve()
data_raw_dir = repo_root / "data" / "raw"

repo_root.mkdir(parents=True, exist_ok=True)
data_raw_dir.mkdir(parents=True, exist_ok=True)

print("Available /kaggle/input folders:")
for p in KAGGLE_INPUT.iterdir():
    print(" -", p)

src_candidates = [
    p for p in KAGGLE_INPUT.rglob("src")
    if p.is_dir() and (p / "data").exists() and (p / "features").exists()
]

if not src_candidates:
    raise FileNotFoundError(
        "Could not find your project src/ folder under /kaggle/input. "
        "Make sure your Kaggle Dataset contains the src directory."
    )

source_src = src_candidates[0]
target_src = repo_root / "src"

if target_src.exists():
    shutil.rmtree(target_src)

shutil.copytree(source_src, target_src)

print("\nCopied src from:", source_src)
print("Copied src to:", target_src)

required_csvs = ["train.csv", "test.csv", "features.csv", "stores.csv"]

for csv_name in required_csvs:
    matches = list(KAGGLE_INPUT.rglob(csv_name))
    if matches:
        shutil.copy2(matches[0], data_raw_dir / csv_name)
        print(f"Copied {csv_name} from:", matches[0])

for zip_path in KAGGLE_INPUT.rglob("*.zip"):
    try:
        with zipfile.ZipFile(zip_path, "r") as z:
            names = z.namelist()
            wanted = [name for name in names if Path(name).name in required_csvs]
            for name in wanted:
                out_name = Path(name).name
                with z.open(name) as src_file, open(data_raw_dir / out_name, "wb") as dst_file:
                    shutil.copyfileobj(src_file, dst_file)
                print(f"Extracted {out_name} from:", zip_path)
    except zipfile.BadZipFile:
        pass

missing = [name for name in required_csvs if not (data_raw_dir / name).exists()]

if missing:
    print("\nFiles currently in data/raw:")
    for p in data_raw_dir.iterdir():
        print(" -", p.name)
    raise FileNotFoundError(f"Missing required raw files: {missing}")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

os.chdir(repo_root)

print("\nRepo root:", repo_root)
print("src exists:", (repo_root / "src").exists())
print("data exists:", data_raw_dir.exists())
print("Raw data files:", sorted(p.name for p in data_raw_dir.iterdir()))

from src.data import load_raw_data, last_n_weeks_split, calendar_aligned_split
from src.features import WalmartBasePreprocessor, WalmartNeuralPreprocessor
from src.datasets import (
    WalmartPrecomputedTrainingWindowDataset,
    WalmartPrecomputedForecastWindowDataset,
    FastTensorDataLoader,
)

Available /kaggle/input folders:
 - /kaggle/input/competitions
 - /kaggle/input/datasets

Copied src from: /kaggle/input/datasets/lukabatilashvili/walmart-code/src
Copied src to: /kaggle/working/Walmart/src
Copied stores.csv from: /kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/stores.csv
Extracted train.csv from: /kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/train.csv.zip
Extracted features.csv from: /kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/features.csv.zip
Extracted test.csv from: /kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/test.csv.zip

Repo root: /kaggle/working/Walmart
src exists: True
data exists: True
Raw data files: ['features.csv', 'stores.csv', 'test.csv', 'train.csv']


In [43]:
DATA_DIR = repo_root / "data" / "raw"

CONTEXT_LENGTH = 52
PREDICTION_LENGTH = 39

BATCH_SIZE = 256
SEED = 42

MLFLOW_EXPERIMENT_NAME = "NBEATS_Training"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Repo root:", repo_root)
print("Data dir:", DATA_DIR)
print("Device:", DEVICE)

Repo root: /kaggle/working/Walmart
Data dir: /kaggle/working/Walmart/data/raw
Device: cuda


In [44]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

## Dagshub/Mlflow initialization

In [45]:
from kaggle_secrets import UserSecretsClient

os.environ["MLFLOW_TRACKING_USERNAME"] = "LukaBatilashvili07"
os.environ["MLFLOW_TRACKING_PASSWORD"] = UserSecretsClient().get_secret("DAGSHUB_TOKEN")

import dagshub
import mlflow

dagshub.init(repo_owner='LukaBatilashvili07', repo_name='walmart-sales-forecasting', mlflow=True)

Initialized MLflow to track repo "LukaBatilashvili07/walmart-sales-forecasting"

Repository LukaBatilashvili07/walmart-sales-forecasting initialized!

## Load Dataset and Time Split

In [46]:
train, test, stores, features = load_raw_data(DATA_DIR)

for df in [train, test, features]:
    df["Date"] = pd.to_datetime(df["Date"])

print("train:", train.shape)
print("test:", test.shape)
print("stores:", stores.shape)
print("features:", features.shape)

print("\nTrain date range:")
print(train["Date"].min(), "->", train["Date"].max())

print("\nTest date range:")
print(test["Date"].min(), "->", test["Date"].max())

display(train.head())
display(test.head())
display(stores.head())
display(features.head())

train: (421570, 5)
test: (115064, 4)
stores: (45, 3)
features: (8190, 12)

Train date range:
2010-02-05 00:00:00 -> 2012-10-26 00:00:00

Test date range:
2012-11-02 00:00:00 -> 2013-07-26 00:00:00


,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,24924.50,False
1,1,1,2010-02-12,46039.49,True
2,1,1,2010-02-19,41595.55,False
3,1,1,2010-02-26,19403.54,False
4,1,1,2010-03-05,21827.90,False


,Store,Dept,Date,IsHoliday
0,1,1,2012-11-02,False
1,1,1,2012-11-09,False
2,1,1,2012-11-16,False
3,1,1,2012-11-23,True
4,1,1,2012-11-30,False


,Store,Type,Size
0,1,A,151315
1,2,A,202307
2,3,B,37392
3,4,A,205863
4,5,B,34875


,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
0,1,2010-02-05,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False
1,1,2010-02-12,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True
2,1,2010-02-19,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,False
3,1,2010-02-26,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,False
4,1,2010-03-05,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,False


In [47]:
assert {"Store", "Dept", "Date", "Weekly_Sales", "IsHoliday"}.issubset(train.columns)
assert {"Store", "Dept", "Date", "IsHoliday"}.issubset(test.columns)
assert {"Store", "Type", "Size"}.issubset(stores.columns)
assert {"Store", "Date", "IsHoliday"}.issubset(features.columns)

print("Raw data checks passed.")

Raw data checks passed.


In [48]:
# split A: last 39 weeks of train

train_raw_part, valid_raw_part = last_n_weeks_split(
    train,
    n_weeks=PREDICTION_LENGTH,
    date_col="Date",
)

print("Last-39 split")
print("train_raw_part:", train_raw_part.shape)
print("valid_raw_part:", valid_raw_part.shape)

assert train_raw_part["Date"].max() < valid_raw_part["Date"].min()
assert valid_raw_part["Date"].nunique() == PREDICTION_LENGTH

print("Last-39 split checks passed.")

Last-39 split
train_raw_part: (305982, 5)
valid_raw_part: (115588, 5)
Last-39 split checks passed.


In [49]:
# split B: calendar-aligned validation

calendar_train_raw_part, calendar_valid_raw_part = calendar_aligned_split(
    train,
    valid_start="2011-11-04",
    valid_end="2012-07-27",
    date_col="Date",
)

print("Calendar-aligned split")
print("calendar_train_raw_part:", calendar_train_raw_part.shape)
print("calendar_valid_raw_part:", calendar_valid_raw_part.shape)

assert calendar_train_raw_part["Date"].max() < calendar_valid_raw_part["Date"].min()
assert calendar_valid_raw_part["Date"].nunique() == PREDICTION_LENGTH

print("Calendar-aligned split checks passed.")

Calendar-aligned split
calendar_train_raw_part: (267184, 5)
calendar_valid_raw_part: (115856, 5)
Calendar-aligned split checks passed.


## Base + neural preprocessing

In [50]:
base_preprocessor = WalmartBasePreprocessor()
base_preprocessor.fit(stores, features)

# last-39
last39_train_base = base_preprocessor.transform(train_raw_part)
last39_valid_base = base_preprocessor.transform(valid_raw_part)

# calendar-aligned
calendar_train_base = base_preprocessor.transform(calendar_train_raw_part)
calendar_valid_base = base_preprocessor.transform(calendar_valid_raw_part)

print("Last-39 base:", last39_train_base.shape, last39_valid_base.shape)
print("Calendar base:", calendar_train_base.shape, calendar_valid_base.shape)

assert len(last39_train_base) == len(train_raw_part)
assert len(last39_valid_base) == len(valid_raw_part)
assert len(calendar_train_base) == len(calendar_train_raw_part)
assert len(calendar_valid_base) == len(calendar_valid_raw_part)

Last-39 base: (305982, 42) (115588, 42)
Calendar base: (267184, 42) (115856, 42)


In [51]:
# last-39 neural preprocessing
last39_neural_preprocessor = WalmartNeuralPreprocessor()
last39_neural_preprocessor.fit(last39_train_base)

last39_train_panel = last39_neural_preprocessor.transform(last39_train_base)
last39_valid_panel = last39_neural_preprocessor.transform(last39_valid_base)

last39_dataset_cols = last39_neural_preprocessor.get_dataset_columns()

# calendar-aligned neural preprocessing
calendar_neural_preprocessor = WalmartNeuralPreprocessor()
calendar_neural_preprocessor.fit(calendar_train_base)

calendar_train_panel = calendar_neural_preprocessor.transform(calendar_train_base)
calendar_valid_panel = calendar_neural_preprocessor.transform(calendar_valid_base)

calendar_dataset_cols = calendar_neural_preprocessor.get_dataset_columns()

print("Last-39 panels:", last39_train_panel.shape, last39_valid_panel.shape)
print("Calendar panels:", calendar_train_panel.shape, calendar_valid_panel.shape)

Last-39 panels: (305982, 66) (115588, 66)
Calendar panels: (267184, 66) (115856, 66)


In [52]:
expected_neural_cols = [
    "series_id",
    "Store_id", "Dept_id", "Type_id",
    "Weekly_Sales_scaled",
    "target_mean", "target_std",
]

neural_panels = {
    "last39_train_panel": last39_train_panel,
    "last39_valid_panel": last39_valid_panel,
    "calendar_train_panel": calendar_train_panel,
    "calendar_valid_panel": calendar_valid_panel,
}

for name, df in neural_panels.items():
    for col in expected_neural_cols:
        assert col in df.columns, f"Missing from {name}: {col}"
    assert df["Weekly_Sales_scaled"].notna().all(), f"NaN target scale in {name}"
    assert (df["target_std"] > 0).all(), f"Non-positive target_std in {name}"

print("Neural preprocessing checks passed.")

Neural preprocessing checks passed.


In [53]:
valid_batch = next(iter(nbeats_last39_data["valid_loader"]))
print("Last-39 VALID batch keys:")
for key, value in valid_batch.items():
    print(" ", key, tuple(value.shape), value.dtype)

train_batch = next(iter(nbeats_last39_data["train_loader"]))
print("\nLast-39 TRAIN batch keys:")
for key, value in train_batch.items():
    print(" ", key, tuple(value.shape), value.dtype)

Last-39 VALID batch keys:
  past_target (256, 52) torch.float32
  future_target (256, 39) torch.float32
  past_known_reals (256, 52, 1) torch.float32
  future_known_reals (256, 39, 1) torch.float32
  static_categoricals (256, 3) torch.int64
  static_reals (256, 1) torch.float32
  target_mean (256,) torch.float32
  target_std (256,) torch.float32
  store (256,) torch.int64
  dept (256,) torch.int64

Last-39 TRAIN batch keys:
  past_target (256, 52) torch.float32
  future_target (256, 39) torch.float32
  past_known_reals (256, 52, 1) torch.float32
  future_known_reals (256, 39, 1) torch.float32
  static_categoricals (256, 3) torch.int64
  static_reals (256, 1) torch.float32
  target_mean (256,) torch.float32
  target_std (256,) torch.float32


In [54]:
# N-BEATS is univariate: the model itself only consumes past_target (see
# forward_model()). IsHoliday is carried through known_future_real_cols only
# so evaluate_model() can compute a real WMAE (not just MAE) — it is never
# fed into the model.

nbeats_calendar_cols = {
    "target_col": calendar_dataset_cols["target_col"],
    "series_col": calendar_dataset_cols["series_col"],
    "static_cat_cols": calendar_dataset_cols["static_cat_cols"],
    "static_real_cols": calendar_dataset_cols["static_real_cols"],
    "known_future_real_cols": ("IsHoliday",),
}

nbeats_last39_cols = {
    "target_col": last39_dataset_cols["target_col"],
    "series_col": last39_dataset_cols["series_col"],
    "static_cat_cols": last39_dataset_cols["static_cat_cols"],
    "static_real_cols": last39_dataset_cols["static_real_cols"],
    "known_future_real_cols": ("IsHoliday",),
}

print(nbeats_calendar_cols)
print(nbeats_last39_cols)

{'target_col': 'Weekly_Sales_scaled', 'series_col': 'series_id', 'static_cat_cols': ['Store_id', 'Dept_id', 'Type_id'], 'static_real_cols': ['Size_scaled'], 'known_future_real_cols': ('IsHoliday',)}
{'target_col': 'Weekly_Sales_scaled', 'series_col': 'series_id', 'static_cat_cols': ['Store_id', 'Dept_id', 'Type_id'], 'static_real_cols': ['Size_scaled'], 'known_future_real_cols': ('IsHoliday',)}


## Dataset and DataLoaders

`make_full_horizon_validation_panel` keeps only Store-Dept series that have the
full 39-week validation horizon, so ragged/short series (missing weeks) don't
break window construction downstream.

In [55]:
def make_full_horizon_validation_panel(
    valid_panel: pd.DataFrame,
    prediction_length: int,
    series_col: str = "series_id",
) -> tuple[pd.DataFrame, pd.Index]:
    group_sizes = valid_panel.groupby(series_col).size()

    full_horizon_series = group_sizes[
        group_sizes == prediction_length
    ].index

    valid_panel_full = valid_panel[
        valid_panel[series_col].isin(full_horizon_series)
    ].copy()

    assert len(valid_panel_full) == len(full_horizon_series) * prediction_length

    return valid_panel_full, full_horizon_series

In [56]:
calendar_valid_panel_full, calendar_full_horizon_series = make_full_horizon_validation_panel(
    calendar_valid_panel,
    prediction_length=PREDICTION_LENGTH,
    series_col=nbeats_calendar_cols["series_col"],
)

last39_valid_panel_full, last39_full_horizon_series = make_full_horizon_validation_panel(
    last39_valid_panel,
    prediction_length=PREDICTION_LENGTH,
    series_col=nbeats_last39_cols["series_col"],
)

print("Calendar validation:")
print("total series:", calendar_valid_panel["series_id"].nunique())
print("full-horizon series:", len(calendar_full_horizon_series))

print("\nLast-39 validation:")
print("total series:", last39_valid_panel["series_id"].nunique())
print("full-horizon series:", len(last39_full_horizon_series))

Calendar validation:
total series: 3233
full-horizon series: 2762

Last-39 validation:
total series: 3204
full-horizon series: 2762


In [57]:
def build_nbeats_data_bundle(
    train_panel: pd.DataFrame,
    valid_panel_full: pd.DataFrame,
    cols: dict,
    batch_size: int = BATCH_SIZE,
) -> dict:
    train_dataset = WalmartPrecomputedTrainingWindowDataset(
        train_panel,
        context_length=CONTEXT_LENGTH,
        prediction_length=PREDICTION_LENGTH,
        target_col=cols["target_col"],
        series_col=cols["series_col"],
        static_cat_cols=cols["static_cat_cols"],
        static_real_cols=cols["static_real_cols"],
        known_future_real_cols=cols["known_future_real_cols"],
        drop_nan_targets=True,
    )

    valid_dataset = WalmartPrecomputedForecastWindowDataset(
        history_df=train_panel,
        future_df=valid_panel_full,
        context_length=CONTEXT_LENGTH,
        prediction_length=PREDICTION_LENGTH,
        target_col=cols["target_col"],
        series_col=cols["series_col"],
        static_cat_cols=cols["static_cat_cols"],
        static_real_cols=cols["static_real_cols"],
        known_future_real_cols=cols["known_future_real_cols"],
    )

    train_loader = FastTensorDataLoader(
        train_dataset.tensors,
        batch_size=batch_size,
        shuffle=True,
    )

    valid_loader = FastTensorDataLoader(
        valid_dataset.tensors,
        batch_size=batch_size,
        shuffle=False,
    )

    assert train_dataset.tensors["past_target"].shape[1] == CONTEXT_LENGTH
    assert train_dataset.tensors["future_target"].shape[1] == PREDICTION_LENGTH

    return {
        "cols": cols,
        "train_dataset": train_dataset,
        "valid_dataset": valid_dataset,
        "train_loader": train_loader,
        "valid_loader": valid_loader,
        "batch_size": batch_size,
    }

In [58]:
nbeats_calendar_data = build_nbeats_data_bundle(
    train_panel=calendar_train_panel,
    valid_panel_full=calendar_valid_panel_full,
    cols=nbeats_calendar_cols,
    batch_size=BATCH_SIZE,
)

nbeats_last39_data = build_nbeats_data_bundle(
    train_panel=last39_train_panel,
    valid_panel_full=last39_valid_panel_full,
    cols=nbeats_last39_cols,
    batch_size=BATCH_SIZE,
)

print("Calendar N-BEATS data:")
print("train windows:", len(nbeats_calendar_data["train_dataset"]))
print("valid forecast series:", len(nbeats_calendar_data["valid_dataset"]))

print("\nLast-39 N-BEATS data:")
print("train windows:", len(nbeats_last39_data["train_dataset"]))
print("valid forecast series:", len(nbeats_last39_data["valid_dataset"]))

Calendar N-BEATS data:
train windows: 2683
valid forecast series: 2762

Last-39 N-BEATS data:
train windows: 37597
valid forecast series: 2762


In [59]:
calendar_batch = next(iter(nbeats_calendar_data["train_loader"]))

print("Calendar train batch:")
for key, value in calendar_batch.items():
    print(key, tuple(value.shape), value.dtype)

Calendar train batch:
past_target (256, 52) torch.float32
future_target (256, 39) torch.float32
past_known_reals (256, 52, 1) torch.float32
future_known_reals (256, 39, 1) torch.float32
static_categoricals (256, 3) torch.int64
static_reals (256, 1) torch.float32
target_mean (256,) torch.float32
target_std (256,) torch.float32


In [60]:
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

with mlflow.start_run(run_name="NBEATS_Preprocessing") as run:
    mlflow.log_param("model_family", "NBEATS")
    mlflow.log_param("context_length", CONTEXT_LENGTH)
    mlflow.log_param("prediction_length", PREDICTION_LENGTH)
    mlflow.log_param("base_preprocessor", "WalmartBasePreprocessor")
    mlflow.log_param("neural_preprocessor", "WalmartNeuralPreprocessor")
    mlflow.log_param("primary_validation_strategy", "calendar_aligned_39_weeks")
    mlflow.log_param("secondary_validation_strategy", "last_39_weeks")
    mlflow.log_param("feature_set", "target_only_univariate")
    mlflow.log_param("forecast_validation", "full_horizon_series_only")

    mlflow.log_metric("raw_train_rows", len(train))
    mlflow.log_metric("raw_test_rows", len(test))

    mlflow.log_metric("last39_train_windows", len(nbeats_last39_data["train_dataset"]))
    mlflow.log_metric("last39_full_horizon_valid_series", len(nbeats_last39_data["valid_dataset"]))

    mlflow.log_metric("calendar_train_windows", len(nbeats_calendar_data["train_dataset"]))
    mlflow.log_metric("calendar_full_horizon_valid_series", len(nbeats_calendar_data["valid_dataset"]))

    nbeats_preprocessing_run_id = run.info.run_id

print("Logged N-BEATS preprocessing run:", nbeats_preprocessing_run_id)

🏃 View run NBEATS_Preprocessing at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/8/runs/b571eafcf9584c7cbaf779405dd8e8fb
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/8
Logged N-BEATS preprocessing run: b571eafcf9584c7cbaf779405dd8e8fb


## Naive baselines

Same seasonal-naive (t-52) and last-value-naive baselines used elsewhere in the
project, computed on the calendar-aligned split so N-BEATS is judged against the
same reference points as the other models.

In [61]:
def compute_wmae(y_true: np.ndarray, y_pred: np.ndarray, is_holiday: np.ndarray) -> float:
    weights = np.where(is_holiday, 5.0, 1.0)
    return float(np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights))


def seasonal_naive_baseline_wmae(history_df: pd.DataFrame, future_df: pd.DataFrame) -> float:
    history = history_df.copy()
    future = future_df.copy()
    history["Date"] = pd.to_datetime(history["Date"])
    future["Date"] = pd.to_datetime(future["Date"])

    history_lookup = history.set_index(["Store", "Dept", "Date"])["Weekly_Sales"]

    preds = []
    for _, row in future.iterrows():
        lookup_date = row["Date"] - pd.Timedelta(weeks=52)
        key = (row["Store"], row["Dept"], lookup_date)
        preds.append(history_lookup.get(key, np.nan))

    future = future.assign(y_pred=preds)
    valid_mask = future["y_pred"].notna() & future["Weekly_Sales"].notna()

    return compute_wmae(
        future.loc[valid_mask, "Weekly_Sales"].to_numpy(),
        future.loc[valid_mask, "y_pred"].to_numpy(),
        future.loc[valid_mask, "IsHoliday"].to_numpy(dtype=bool),
    )


def last_value_naive_baseline_wmae(history_df: pd.DataFrame, future_df: pd.DataFrame) -> float:
    history = history_df.copy()
    future = future_df.copy()
    history["Date"] = pd.to_datetime(history["Date"])
    future["Date"] = pd.to_datetime(future["Date"])

    last_known = history.sort_values("Date").groupby(["Store", "Dept"])["Weekly_Sales"].last()

    future = future.merge(last_known.rename("y_pred"), on=["Store", "Dept"], how="left")
    valid_mask = future["y_pred"].notna() & future["Weekly_Sales"].notna()

    return compute_wmae(
        future.loc[valid_mask, "Weekly_Sales"].to_numpy(),
        future.loc[valid_mask, "y_pred"].to_numpy(),
        future.loc[valid_mask, "IsHoliday"].to_numpy(dtype=bool),
    )


seasonal_naive_wmae = seasonal_naive_baseline_wmae(calendar_train_raw_part, calendar_valid_raw_part)
last_value_naive_wmae = last_value_naive_baseline_wmae(calendar_train_raw_part, calendar_valid_raw_part)

print("Seasonal naive (t-52) WMAE:", seasonal_naive_wmae)
print("Last-value naive WMAE:", last_value_naive_wmae)

with mlflow.start_run(run_name="NBEATS_Baselines"):
    mlflow.log_metric("seasonal_naive_wmae", seasonal_naive_wmae)
    mlflow.log_metric("last_value_naive_wmae", last_value_naive_wmae)

Seasonal naive (t-52) WMAE: 2064.305212373069
Last-value naive WMAE: 3863.1462811643037
🏃 View run NBEATS_Baselines at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/8/runs/d4daa576abf3481ba375bba7e7b05ab5
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/8


## Evaluation utilities

In [62]:
def weighted_mae_np(y_true, y_pred, is_holiday):
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=np.float64).reshape(-1)
    is_holiday = np.asarray(is_holiday).reshape(-1).astype(bool)

    weights = np.where(is_holiday, 5.0, 1.0)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)


def inverse_scale_torch(y_scaled, target_mean, target_std):
    return y_scaled * target_std.unsqueeze(1) + target_mean.unsqueeze(1)


def move_batch_to_device(batch, device):
    return {
        key: value.to(device) if torch.is_tensor(value) else value
        for key, value in batch.items()
    }


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def make_loss_fn(loss_name: str):
    loss_name = loss_name.lower()
    if loss_name == "mse":
        return nn.MSELoss()
    if loss_name in ["mae", "l1"]:
        return nn.L1Loss()
    if loss_name == "huber":
        return nn.HuberLoss(delta=1.0)
    raise ValueError(f"Unknown loss_name: {loss_name}")

## N-BEATS Model Definitions

Generic N-BEATS: stacks of blocks, each producing a backcast (subtracted from
the residual) and a forecast (summed into the total). No fixed basis functions everything is learned freely.

In [63]:
class NBeatsBlock(nn.Module):
    def __init__(self, context_length: int, prediction_length: int, hidden_units: int, num_layers: int):
        super().__init__()
        self.context_length = context_length
        self.prediction_length = prediction_length

        layers = [nn.Linear(context_length, hidden_units), nn.ReLU()]
        for _ in range(num_layers - 1):
            layers += [nn.Linear(hidden_units, hidden_units), nn.ReLU()]
        self.fc_stack = nn.Sequential(*layers)

        self.backcast_layer = nn.Linear(hidden_units, context_length)
        self.forecast_layer = nn.Linear(hidden_units, prediction_length)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        h = self.fc_stack(x)
        return self.backcast_layer(h), self.forecast_layer(h)


class NBeatsStack(nn.Module):
    def __init__(self, context_length: int, prediction_length: int, hidden_units: int, num_layers: int, num_blocks: int):
        super().__init__()
        self.blocks = nn.ModuleList([
            NBeatsBlock(context_length, prediction_length, hidden_units, num_layers)
            for _ in range(num_blocks)
        ])

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        residual = x
        stack_forecast = torch.zeros(
            x.shape[0], self.blocks[0].prediction_length, device=x.device, dtype=x.dtype
        )
        for block in self.blocks:
            backcast, forecast = block(residual)
            residual = residual - backcast
            stack_forecast = stack_forecast + forecast
        return residual, stack_forecast


class NBeats(nn.Module):
    def __init__(
        self,
        context_length: int = 52,
        prediction_length: int = 39,
        num_stacks: int = 3,
        num_blocks_per_stack: int = 3,
        hidden_units: int = 256,
        num_layers: int = 4,
    ):
        super().__init__()
        self.stacks = nn.ModuleList([
            NBeatsStack(context_length, prediction_length, hidden_units, num_layers, num_blocks_per_stack)
            for _ in range(num_stacks)
        ])

    def forward(self, past_target: torch.Tensor) -> torch.Tensor:
        residual = past_target
        total_forecast = torch.zeros(
            past_target.shape[0],
            self.stacks[0].blocks[0].prediction_length,
            device=past_target.device,
            dtype=past_target.dtype,
        )
        for stack in self.stacks:
            residual, stack_forecast = stack(residual)
            total_forecast = total_forecast + stack_forecast
        return total_forecast

## Training utilities

In [64]:
def forward_model(model, batch):
    return model(past_target=batch["past_target"])


def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0.0
    total_items = 0

    for batch in loader:
        batch = move_batch_to_device(batch, device)
        optimizer.zero_grad(set_to_none=True)

        preds_scaled = forward_model(model, batch)
        target_scaled = batch["future_target"]

        loss = loss_fn(preds_scaled, target_scaled)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        n_items = target_scaled.numel()
        total_loss += loss.item() * n_items
        total_items += n_items

    return total_loss / total_items

In [65]:
@torch.no_grad()
def evaluate_model(model, loader, loss_fn, device, holiday_col_index: int = 0):
    model.eval()
    total_loss = 0.0
    total_items = 0

    all_true, all_pred, all_holiday = [], [], []

    for batch in loader:
        batch = move_batch_to_device(batch, device)

        preds_scaled = forward_model(model, batch)
        target_scaled = batch["future_target"]

        loss = loss_fn(preds_scaled, target_scaled)
        n_items = target_scaled.numel()
        total_loss += loss.item() * n_items
        total_items += n_items

        preds_original = inverse_scale_torch(preds_scaled, batch["target_mean"], batch["target_std"])
        target_original = inverse_scale_torch(target_scaled, batch["target_mean"], batch["target_std"])

        all_pred.append(preds_original.detach().cpu().numpy())
        all_true.append(target_original.detach().cpu().numpy())

        # IsHoliday is not a standalone batch key it's stacked into
        # future_known_reals (shape: batch, prediction_length, num_known_reals)
        # alongside every other column in known_future_real_cols. Since
        # known_future_real_cols=("IsHoliday",), it sits at index 0.
        if "future_known_reals" not in batch:
            raise KeyError(
                "future_known_reals missing from batch — check known_future_real_cols wiring in Cell 17."
            )
        holiday_flags = batch["future_known_reals"][..., holiday_col_index]
        all_holiday.append(holiday_flags.detach().cpu().numpy())

    y_pred = np.concatenate(all_pred, axis=0)
    y_true = np.concatenate(all_true, axis=0)
    is_holiday = np.concatenate(all_holiday, axis=0)

    valid_loss = total_loss / total_items
    valid_wmae = weighted_mae_np(y_true, y_pred, is_holiday)
    valid_mae = np.mean(np.abs(y_true.reshape(-1) - y_pred.reshape(-1)))

    return {"valid_loss": valid_loss, "valid_wmae": valid_wmae, "valid_mae": valid_mae}

In [66]:
def fit_model(model, train_loader, valid_loader, optimizer, loss_fn, device, epochs: int, metric_prefix: str = ""):
    best_state_dict = None
    best_valid_wmae = float("inf")
    best_epoch = None
    history = []
    prefix = f"{metric_prefix}_" if metric_prefix else ""

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
        valid_metrics = evaluate_model(model, valid_loader, loss_fn, device)

        history.append({"epoch": epoch, "train_loss": train_loss, **valid_metrics})

        if mlflow.active_run() is not None:
            mlflow.log_metric(f"{prefix}train_loss", train_loss, step=epoch)
            mlflow.log_metric(f"{prefix}valid_loss", valid_metrics["valid_loss"], step=epoch)
            mlflow.log_metric(f"{prefix}valid_wmae", valid_metrics["valid_wmae"], step=epoch)
            mlflow.log_metric(f"{prefix}valid_mae", valid_metrics["valid_mae"], step=epoch)

        if valid_metrics["valid_wmae"] < best_valid_wmae:
            best_valid_wmae = valid_metrics["valid_wmae"]
            best_epoch = epoch
            best_state_dict = deepcopy(model.state_dict())

        print(
            f"Epoch {epoch:03d} | train_loss={train_loss:.5f} | "
            f"valid_wmae={valid_metrics['valid_wmae']:.2f} | valid_mae={valid_metrics['valid_mae']:.2f}"
        )

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    return {
        "model": model,
        "history": pd.DataFrame(history),
        "best_valid_wmae": best_valid_wmae,
        "best_epoch": best_epoch,
        "best_state_dict": best_state_dict,
    }

## N-BEATS Experiments: Last-39 Validation

In [67]:
NBEATS_BASELINE_CONFIG = {
    "model": "NBeats",
    "model_variant": "nbeats_generic",
    "validation_strategy": "last_39_weeks",
    "context_length": CONTEXT_LENGTH,
    "prediction_length": PREDICTION_LENGTH,
    "num_stacks": 3,
    "num_blocks_per_stack": 3,
    "hidden_units": 256,
    "num_layers": 4,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "loss_name": "huber",
    "batch_size": BATCH_SIZE,
    "epochs": 60,
    "patience": 10,
}

In [ ]:
NBEATS_LAST39_GRID = [
    {"num_stacks": 2, "num_blocks_per_stack": 2, "hidden_units": 64,  "num_layers": 2, "lr": 1e-3, "weight_decay": 1e-4, "loss_name": "huber"},
    {"num_stacks": 3, "num_blocks_per_stack": 3, "hidden_units": 128, "num_layers": 3, "lr": 1e-3, "weight_decay": 1e-4, "loss_name": "huber"},
    {"num_stacks": 3, "num_blocks_per_stack": 3, "hidden_units": 256, "num_layers": 4, "lr": 5e-4, "weight_decay": 1e-4, "loss_name": "huber"},
    {"num_stacks": 4, "num_blocks_per_stack": 3, "hidden_units": 256, "num_layers": 4, "lr": 5e-4, "weight_decay": 1e-3, "loss_name": "mse"},
    {"num_stacks": 5, "num_blocks_per_stack": 4, "hidden_units": 512, "num_layers": 4, "lr": 1e-4, "weight_decay": 1e-4, "loss_name": "huber"},
]

set_seed(SEED)
nbeats_last39_results = []

for i, config in enumerate(NBEATS_LAST39_GRID, start=1):
    config = {**config, "context_length": CONTEXT_LENGTH, "prediction_length": PREDICTION_LENGTH,
              "batch_size": BATCH_SIZE, "epochs": 60, "patience": 10}

    run_name = (
        f"NBEATS_Last39_"
        f"stacks{config['num_stacks']}_blocks{config['num_blocks_per_stack']}_"
        f"hidden{config['hidden_units']}_layers{config['num_layers']}_"
        f"lr{config['lr']}_wd{config['weight_decay']}_{config['loss_name']}"
    )

    model = NBeats(
        context_length=CONTEXT_LENGTH,
        prediction_length=PREDICTION_LENGTH,
        num_stacks=config["num_stacks"],
        num_blocks_per_stack=config["num_blocks_per_stack"],
        hidden_units=config["hidden_units"],
        num_layers=config["num_layers"],
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
    loss_fn = make_loss_fn(config["loss_name"])

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_params(config)
        result = fit_model(
            model=model,
            train_loader=nbeats_last39_data["train_loader"],
            valid_loader=nbeats_last39_data["valid_loader"],
            optimizer=optimizer,
            loss_fn=loss_fn,
            device=DEVICE,
            epochs=config["epochs"],
            metric_prefix="last39",
        )
        mlflow.log_metric("last39_best_valid_wmae", result["best_valid_wmae"])
        mlflow.log_metric("last39_best_epoch", result["best_epoch"])

    print(run_name, "->", result["best_valid_wmae"])
    nbeats_last39_results.append({"run_name": run_name, "run_id": run.info.run_id, **config,
                                    "best_valid_wmae": result["best_valid_wmae"], "best_epoch": result["best_epoch"]})

nbeats_last39_results_df = pd.DataFrame(nbeats_last39_results).sort_values("best_valid_wmae").reset_index(drop=True)
display(nbeats_last39_results_df)

Epoch 001 | train_loss=0.20216 | valid_wmae=2519.16 | valid_mae=2504.34
Epoch 002 | train_loss=0.16299 | valid_wmae=2461.15 | valid_mae=2447.84
Epoch 003 | train_loss=0.15847 | valid_wmae=2443.42 | valid_mae=2428.27
Epoch 004 | train_loss=0.15598 | valid_wmae=2466.28 | valid_mae=2443.65
Epoch 005 | train_loss=0.15440 | valid_wmae=2420.42 | valid_mae=2418.38
Epoch 006 | train_loss=0.15258 | valid_wmae=2425.64 | valid_mae=2402.60
Epoch 007 | train_loss=0.15142 | valid_wmae=2445.23 | valid_mae=2438.01
Epoch 008 | train_loss=0.15038 | valid_wmae=2436.20 | valid_mae=2428.30
Epoch 009 | train_loss=0.14947 | valid_wmae=2418.99 | valid_mae=2414.26
Epoch 010 | train_loss=0.14858 | valid_wmae=2430.67 | valid_mae=2414.15
Epoch 011 | train_loss=0.14786 | valid_wmae=2427.56 | valid_mae=2417.75
Epoch 012 | train_loss=0.14710 | valid_wmae=2431.30 | valid_mae=2421.43
Epoch 013 | train_loss=0.14666 | valid_wmae=2420.59 | valid_mae=2396.68
Epoch 014 | train_loss=0.14602 | valid_wmae=2433.88 | valid_mae=

## N-BEATS Experiments: Calendar-aligned

In [69]:
top_configs = nbeats_last39_results_df.sort_values("best_valid_wmae").head(3).copy()

set_seed(SEED)
nbeats_calendar_results = []

for _, row in top_configs.iterrows():
    config = {
        "num_stacks": int(row["num_stacks"]),
        "num_blocks_per_stack": int(row["num_blocks_per_stack"]),
        "hidden_units": int(row["hidden_units"]),
        "num_layers": int(row["num_layers"]),
        "lr": row["lr"],
        "weight_decay": row["weight_decay"],
        "loss_name": row["loss_name"],
        "context_length": CONTEXT_LENGTH,
        "prediction_length": PREDICTION_LENGTH,
        "batch_size": BATCH_SIZE,
        "epochs": 60,
        "patience": 15,
        "source_last39_run_id": row["run_id"],
        "source_last39_best_valid_wmae": row["best_valid_wmae"],
    }

    run_name = (
        f"NBEATS_CalendarAligned_"
        f"stacks{config['num_stacks']}_blocks{config['num_blocks_per_stack']}_"
        f"hidden{config['hidden_units']}_layers{config['num_layers']}_"
        f"lr{config['lr']}_wd{config['weight_decay']}_{config['loss_name']}"
    )

    model = NBeats(
        context_length=CONTEXT_LENGTH,
        prediction_length=PREDICTION_LENGTH,
        num_stacks=config["num_stacks"],
        num_blocks_per_stack=config["num_blocks_per_stack"],
        hidden_units=config["hidden_units"],
        num_layers=config["num_layers"],
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
    loss_fn = make_loss_fn(config["loss_name"])

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_params(config)
        result = fit_model(
            model=model,
            train_loader=nbeats_calendar_data["train_loader"],
            valid_loader=nbeats_calendar_data["valid_loader"],
            optimizer=optimizer,
            loss_fn=loss_fn,
            device=DEVICE,
            epochs=config["epochs"],
            metric_prefix="calendar",
        )
        mlflow.log_metric("calendar_best_valid_wmae", result["best_valid_wmae"])
        mlflow.log_metric("calendar_best_epoch", result["best_epoch"])

    print(run_name, "->", result["best_valid_wmae"])
    nbeats_calendar_results.append({"run_name": run_name, "run_id": run.info.run_id, **config,
                                      "best_valid_wmae": result["best_valid_wmae"], "best_epoch": result["best_epoch"]})

nbeats_calendar_results_df = pd.DataFrame(nbeats_calendar_results).sort_values("best_valid_wmae").reset_index(drop=True)
display(nbeats_calendar_results_df)

Epoch 024 | train_loss=0.09480 | valid_wmae=3606.62 | valid_mae=3208.04
Epoch 025 | train_loss=0.09336 | valid_wmae=3607.68 | valid_mae=3200.42
Epoch 026 | train_loss=0.09124 | valid_wmae=3625.66 | valid_mae=3253.00
Epoch 027 | train_loss=0.09175 | valid_wmae=3620.78 | valid_mae=3224.46
Epoch 028 | train_loss=0.09031 | valid_wmae=3612.06 | valid_mae=3208.98
Epoch 029 | train_loss=0.08977 | valid_wmae=3639.51 | valid_mae=3281.71
Epoch 030 | train_loss=0.08774 | valid_wmae=3626.02 | valid_mae=3279.97
Epoch 031 | train_loss=0.08657 | valid_wmae=3646.31 | valid_mae=3264.36
Epoch 032 | train_loss=0.08474 | valid_wmae=3628.23 | valid_mae=3241.14
Epoch 033 | train_loss=0.08391 | valid_wmae=3603.65 | valid_mae=3233.02
Epoch 034 | train_loss=0.08196 | valid_wmae=3626.72 | valid_mae=3277.85
Epoch 035 | train_loss=0.08100 | valid_wmae=3674.34 | valid_mae=3313.07
Epoch 036 | train_loss=0.08055 | valid_wmae=3688.42 | valid_mae=3317.41
Epoch 037 | train_loss=0.07939 | valid_wmae=3674.81 | valid_mae=

,run_name,run_id,num_stacks,num_blocks_per_stack,hidden_units,num_layers,lr,weight_decay,loss_name,context_length,prediction_length,batch_size,epochs,patience,source_last39_run_id,source_last39_best_valid_wmae,best_valid_wmae,best_epoch
0,NBEATS_CalendarAligned_stacks5_blocks4_hidden5...,cb77f2c04c494e73ba414aa708bb7e69,5,4,512,4,0.0001,0.0001,huber,52,39,256,60,15,c6df649b28944f1daf69836051039465,2352.405823,3422.162691,5
1,NBEATS_CalendarAligned_stacks3_blocks3_hidden1...,3b78b99d6d49493e822059972f2028c3,3,3,128,3,0.0010,0.0001,huber,52,39,256,60,15,e2e074ca02a246399e0815c1838c992e,2344.900477,3477.933869,4
2,NBEATS_CalendarAligned_stacks2_blocks2_hidden6...,5c751a24af04405bbfe07aa276adc5a6,2,2,64,2,0.0010,0.0001,huber,52,39,256,60,15,44b014e21a3548c6afae31af08778cf6,2365.414940,3481.584832,10


In [70]:
runs = mlflow.search_runs(experiment_names=[MLFLOW_EXPERIMENT_NAME], output_format="pandas")

calendar_runs = runs[
    runs["tags.mlflow.runName"].astype(str).str.contains("NBEATS_CalendarAligned", na=False)
].copy()

cols = [
    "tags.mlflow.runName", "run_id",
    "params.num_stacks", "params.num_blocks_per_stack", "params.hidden_units", "params.num_layers",
    "params.lr", "params.weight_decay", "params.loss_name",
    "metrics.calendar_best_valid_wmae", "metrics.calendar_best_epoch",
]
existing_cols = [c for c in cols if c in calendar_runs.columns]

nbeats_calendar_leaderboard_df = (
    calendar_runs[existing_cols]
    .dropna(subset=["metrics.calendar_best_valid_wmae"])
    .sort_values("metrics.calendar_best_valid_wmae")
    .reset_index(drop=True)
)

display(nbeats_calendar_leaderboard_df)

,tags.mlflow.runName,run_id,params.num_stacks,params.num_blocks_per_stack,params.hidden_units,params.num_layers,params.lr,params.weight_decay,params.loss_name,metrics.calendar_best_valid_wmae,metrics.calendar_best_epoch
0,NBEATS_CalendarAligned_stacks5_blocks4_hidden5...,f6c4f17dea9748b6adada1cbd010803c,5,4,512,4,0.0001,0.0001,huber,3027.122638,6.0
1,NBEATS_CalendarAligned_stacks3_blocks3_hidden1...,2abf8bfb5f8848e98d9720111fc9fac4,3,3,128,3,0.001,0.0001,huber,3054.776083,4.0
2,NBEATS_CalendarAligned_stacks2_blocks2_hidden6...,8004788a91e84f5387ba4cdebdaf8384,2,2,64,2,0.001,0.0001,huber,3057.838101,10.0
3,NBEATS_CalendarAligned_stacks5_blocks4_hidden5...,cb77f2c04c494e73ba414aa708bb7e69,5,4,512,4,0.0001,0.0001,huber,3422.162691,5.0
4,NBEATS_CalendarAligned_stacks3_blocks3_hidden1...,3b78b99d6d49493e822059972f2028c3,3,3,128,3,0.001,0.0001,huber,3477.933869,4.0
5,NBEATS_CalendarAligned_stacks2_blocks2_hidden6...,5c751a24af04405bbfe07aa276adc5a6,2,2,64,2,0.001,0.0001,huber,3481.584832,10.0


In [71]:
train_dataset = nbeats_calendar_data["train_dataset"]
valid_dataset = nbeats_calendar_data["valid_dataset"]
train_loader = nbeats_calendar_data["train_loader"]
valid_loader = nbeats_calendar_data["valid_loader"]

valid_index_df = valid_dataset.get_future_index()

print("train windows:", len(train_dataset))
print("valid forecast series:", len(valid_dataset))
print("train batches:", sum(1 for _ in train_loader))
print("valid batches:", sum(1 for _ in valid_loader))
print("valid date range:", valid_index_df["Date"].min(), valid_index_df["Date"].max())
print("valid rows evaluated:", len(valid_index_df))
print("unique valid series_id:", valid_index_df.groupby(["Store","Dept"]).ngroups)

train windows: 2683
valid forecast series: 2762
train batches: 11
valid batches: 11
valid date range: 2011-11-04 00:00:00 2012-07-27 00:00:00
valid rows evaluated: 107718
unique valid series_id: 2762


In [76]:
# Full-data preprocessing for final submission generation.
# Fits on the ENTIRE train.csv (not a split) so the final model can be
# refit on all 143 weeks, and test.csv gets transformed the same way.

final_base_preprocessor = WalmartBasePreprocessor()
final_base_preprocessor.fit(stores, features)

full_train_base = final_base_preprocessor.transform(train)

# test.csv has real per-series gaps (some Store-Dept combos report fewer
# than 39 weeks e.g. Store=10/Dept=18 has only 29). The forecast dataset
# requires exactly PREDICTION_LENGTH rows per series, so build a complete
# Store x Dept x Date grid first; extra synthetic rows get dropped later
# by the final test_keys merge, which only keeps rows that really exist
# in test.csv.

test_dates = pd.Index(sorted(test["Date"].unique()))
assert len(test_dates) == PREDICTION_LENGTH, (
    f"Expected {PREDICTION_LENGTH} unique test dates, got {len(test_dates)}"
)

date_holiday_map = test.groupby("Date")["IsHoliday"].first()

store_dept_pairs = test[["Store", "Dept"]].drop_duplicates()

full_test_grid = (
    store_dept_pairs.assign(key=1)
    .merge(pd.DataFrame({"Date": test_dates, "key": 1}), on="key")
    .drop(columns="key")
)
full_test_grid["IsHoliday"] = full_test_grid["Date"].map(date_holiday_map)
assert full_test_grid["IsHoliday"].notna().all()

print("Full test grid rows:", full_test_grid.shape[0],
      "(", store_dept_pairs.shape[0], "series x", len(test_dates), "weeks )")
print("Raw test.csv rows:", test.shape[0], "(", test.shape[0] - full_test_grid.shape[0], "fewer — confirms gaps)")

full_test_base = final_base_preprocessor.transform(full_test_grid)

final_neural_preprocessor = WalmartNeuralPreprocessor()
final_neural_preprocessor.fit(full_train_base)

full_train_panel = final_neural_preprocessor.transform(full_train_base)
full_test_grid_panel = final_neural_preprocessor.transform(full_test_base)

final_dataset_cols = final_neural_preprocessor.get_dataset_columns()

final_nbeats_cols = {
    "target_col": final_dataset_cols["target_col"],
    "series_col": final_dataset_cols["series_col"],
    "static_cat_cols": final_dataset_cols["static_cat_cols"],
    "static_real_cols": final_dataset_cols["static_real_cols"],
    "known_future_real_cols": ("IsHoliday",),
}

print("Full train panel:", full_train_panel.shape)
print("Full test grid panel:", full_test_grid_panel.shape)


@torch.no_grad()
def predict_nbeats(model, loader, device):
    model.eval()
    all_preds = []
    for batch in loader:
        batch = move_batch_to_device(batch, device)
        preds_scaled = forward_model(model, batch)
        preds_original = inverse_scale_torch(preds_scaled, batch["target_mean"], batch["target_std"])
        all_preds.append(preds_original.detach().cpu().numpy())
    return np.concatenate(all_preds, axis=0)

Full test grid rows: 123591 ( 3169 series x 39 weeks )
Raw test.csv rows: 115064 ( -8527 fewer — confirms gaps)
Full train panel: (421570, 66)
Full test grid panel: (123591, 65)


# Kaggle Check

In [77]:
def train_and_generate_submission(run_name, config, train_loader, valid_loader, epochs):
    set_seed(SEED)

    model_kwargs = dict(
        num_stacks=int(config["num_stacks"]),
        num_blocks_per_stack=int(config["num_blocks_per_stack"]),
        hidden_units=int(config["hidden_units"]),
        num_layers=int(config["num_layers"]),
    )

    model = NBeats(
        context_length=CONTEXT_LENGTH,
        prediction_length=PREDICTION_LENGTH,
        **model_kwargs,
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=float(config["lr"]),
        weight_decay=float(config["weight_decay"]),
    )
    loss_fn = make_loss_fn(config["loss_name"])

    result = fit_model(
        model=model,
        train_loader=train_loader,
        valid_loader=valid_loader,
        optimizer=optimizer,
        loss_fn=loss_fn,
        device=DEVICE,
        epochs=epochs,
        metric_prefix="quicktest",
    )

    print(run_name, "-> valid_wmae:", result["best_valid_wmae"])

    test_dataset = WalmartPrecomputedForecastWindowDataset(
        history_df=full_train_panel,
        future_df=full_test_grid_panel,
        context_length=CONTEXT_LENGTH,
        prediction_length=PREDICTION_LENGTH,
        **final_nbeats_cols,
    )
    test_loader = FastTensorDataLoader(test_dataset.tensors, batch_size=BATCH_SIZE, shuffle=False)

    preds_matrix = predict_nbeats(model, test_loader, DEVICE)

    test_index_df = test_dataset.get_future_index().reset_index(drop=True)
    pred_df = test_index_df.copy()
    pred_df["Weekly_Sales"] = preds_matrix.reshape(-1)
    pred_df["Weekly_Sales"] = pred_df["Weekly_Sales"].clip(lower=0)

    test_keys = test[["Store", "Dept", "Date"]].copy()
    test_keys["Date"] = pd.to_datetime(test_keys["Date"])

    submission_df = test_keys.merge(pred_df, on=["Store", "Dept", "Date"], how="left")
    assert submission_df["Weekly_Sales"].notna().all()

    submission_df["Id"] = (
        submission_df["Store"].astype(str) + "_"
        + submission_df["Dept"].astype(str) + "_"
        + submission_df["Date"].dt.strftime("%Y-%m-%d")
    )
    submission_df = submission_df[["Id", "Weekly_Sales"]]

    filename = f"{run_name}.csv"
    submission_df.to_csv(filename, index=False)
    print("saved:", filename)

    return model, result, submission_df

In [78]:
best_calendar_row = nbeats_calendar_leaderboard_df.iloc[0]

calendar_model, calendar_result, calendar_submission_df = train_and_generate_submission(
    run_name=best_calendar_row["tags.mlflow.runName"],
    config={
        "num_stacks": best_calendar_row["params.num_stacks"],
        "num_blocks_per_stack": best_calendar_row["params.num_blocks_per_stack"],
        "hidden_units": best_calendar_row["params.hidden_units"],
        "num_layers": best_calendar_row["params.num_layers"],
        "lr": best_calendar_row["params.lr"],
        "weight_decay": best_calendar_row["params.weight_decay"],
        "loss_name": best_calendar_row["params.loss_name"],
    },
    train_loader=nbeats_calendar_data["train_loader"],
    valid_loader=nbeats_calendar_data["valid_loader"],
    epochs=60,
)

Epoch 001 | train_loss=0.26261 | valid_wmae=3493.61 | valid_mae=3027.63
Epoch 002 | train_loss=0.19429 | valid_wmae=3455.29 | valid_mae=3066.32
Epoch 003 | train_loss=0.16391 | valid_wmae=3432.31 | valid_mae=3039.47
Epoch 004 | train_loss=0.14963 | valid_wmae=3435.97 | valid_mae=3045.20
Epoch 005 | train_loss=0.14118 | valid_wmae=3434.93 | valid_mae=3043.16
Epoch 006 | train_loss=0.13552 | valid_wmae=3440.80 | valid_mae=3036.23
Epoch 007 | train_loss=0.13142 | valid_wmae=3446.32 | valid_mae=3046.75
Epoch 008 | train_loss=0.12835 | valid_wmae=3450.68 | valid_mae=3050.84
Epoch 009 | train_loss=0.12479 | valid_wmae=3429.57 | valid_mae=3035.48
Epoch 010 | train_loss=0.12268 | valid_wmae=3459.26 | valid_mae=3064.53
Epoch 011 | train_loss=0.12042 | valid_wmae=3483.41 | valid_mae=3075.48
Epoch 012 | train_loss=0.11844 | valid_wmae=3477.62 | valid_mae=3067.79
Epoch 013 | train_loss=0.11651 | valid_wmae=3504.58 | valid_mae=3084.86
Epoch 014 | train_loss=0.11462 | valid_wmae=3484.64 | valid_mae=

In [82]:
runs = mlflow.search_runs(experiment_names=[MLFLOW_EXPERIMENT_NAME], output_format="pandas")

last39_runs = runs[
    runs["tags.mlflow.runName"].astype(str).str.contains("NBEATS_Last39_", na=False)
    & ~runs["tags.mlflow.runName"].astype(str).str.contains("Baseline", na=False)
].copy()

cols = [
    "tags.mlflow.runName", "run_id",
    "params.num_stacks", "params.num_blocks_per_stack", "params.hidden_units", "params.num_layers",
    "params.lr", "params.weight_decay", "params.loss_name",
    "metrics.last39_best_valid_wmae", "metrics.last39_best_epoch",
]
existing_cols = [c for c in cols if c in last39_runs.columns]

nbeats_last39_leaderboard_df = (
    last39_runs[existing_cols]
    .dropna(subset=["metrics.last39_best_valid_wmae"])
    .sort_values("metrics.last39_best_valid_wmae")
    .reset_index(drop=True)
)

display(nbeats_last39_leaderboard_df)

,tags.mlflow.runName,run_id,params.num_stacks,params.num_blocks_per_stack,params.hidden_units,params.num_layers,params.lr,params.weight_decay,params.loss_name,metrics.last39_best_valid_wmae,metrics.last39_best_epoch
0,NBEATS_Last39_stacks3_blocks3_hidden128_layers...,69335b157c4c4c53ac69178b943f82bc,3,3,128,3,0.001,0.0001,huber,2333.892242,29.0
1,NBEATS_Last39_stacks3_blocks3_hidden128_layers...,e2e074ca02a246399e0815c1838c992e,3,3,128,3,0.001,0.0001,huber,2344.900477,27.0
2,NBEATS_Last39_stacks5_blocks4_hidden512_layers...,e04ffa6b111a48c4a76c533e23157544,5,4,512,4,0.0001,0.0001,huber,2351.943446,16.0
3,NBEATS_Last39_stacks5_blocks4_hidden512_layers...,c6df649b28944f1daf69836051039465,5,4,512,4,0.0001,0.0001,huber,2352.405823,16.0
4,NBEATS_Last39_stacks2_blocks2_hidden64_layers2...,44b014e21a3548c6afae31af08778cf6,2,2,64,2,0.001,0.0001,huber,2365.414940,59.0
5,NBEATS_Last39_stacks3_blocks3_hidden256_layers...,7691e95b52724c559bee4afe9ccc0b17,3,3,256,4,0.0005,0.0001,huber,2376.341164,6.0
6,NBEATS_Last39_stacks2_blocks2_hidden64_layers2...,f04220212c99472d89d1d3c8c55294c3,2,2,64,2,0.001,0.0001,huber,2379.757243,59.0
7,NBEATS_Last39_stacks3_blocks3_hidden256_layers...,8c541cd9658748d098ec308321f4d32d,3,3,256,4,0.0005,0.0001,huber,2380.708641,45.0
8,NBEATS_Last39_stacks4_blocks3_hidden256_layers...,0a17a44c86ba4731930b862e73d40f1b,4,3,256,4,0.0005,0.001,mse,2406.640875,45.0
9,NBEATS_Last39_stacks4_blocks3_hidden256_layers...,16cf851c4b2d4c4cb76814178169efd2,4,3,256,4,0.0005,0.001,mse,2423.288614,50.0


In [83]:
best_last39_row = nbeats_last39_leaderboard_df.iloc[0]

last39_model, last39_result, last39_submission_df = train_and_generate_submission(
    run_name=best_last39_row["tags.mlflow.runName"],
    config={
        "num_stacks": best_last39_row["params.num_stacks"],
        "num_blocks_per_stack": best_last39_row["params.num_blocks_per_stack"],
        "hidden_units": best_last39_row["params.hidden_units"],
        "num_layers": best_last39_row["params.num_layers"],
        "lr": best_last39_row["params.lr"],
        "weight_decay": best_last39_row["params.weight_decay"],
        "loss_name": best_last39_row["params.loss_name"],
    },
    train_loader=nbeats_last39_data["train_loader"],
    valid_loader=nbeats_last39_data["valid_loader"],
    epochs=60,
)

Epoch 001 | train_loss=0.17986 | valid_wmae=2468.29 | valid_mae=2460.60
Epoch 002 | train_loss=0.15525 | valid_wmae=2392.32 | valid_mae=2395.40
Epoch 003 | train_loss=0.15183 | valid_wmae=2368.69 | valid_mae=2370.89
Epoch 004 | train_loss=0.14959 | valid_wmae=2385.54 | valid_mae=2377.88
Epoch 005 | train_loss=0.14782 | valid_wmae=2373.35 | valid_mae=2372.00
Epoch 006 | train_loss=0.14634 | valid_wmae=2421.63 | valid_mae=2415.97
Epoch 007 | train_loss=0.14543 | valid_wmae=2394.46 | valid_mae=2387.61
Epoch 008 | train_loss=0.14397 | valid_wmae=2474.59 | valid_mae=2447.15
Epoch 009 | train_loss=0.14312 | valid_wmae=2371.47 | valid_mae=2370.93
Epoch 010 | train_loss=0.14212 | valid_wmae=2361.30 | valid_mae=2361.58
Epoch 011 | train_loss=0.14094 | valid_wmae=2396.44 | valid_mae=2381.48
Epoch 012 | train_loss=0.14045 | valid_wmae=2365.39 | valid_mae=2363.12
Epoch 013 | train_loss=0.13953 | valid_wmae=2349.85 | valid_mae=2359.01
Epoch 014 | train_loss=0.13861 | valid_wmae=2381.25 | valid_mae=

## Final training

In [90]:
BEST_NBEATS_FINAL_CONFIG = {
    "num_stacks": nbeats_calendar_leaderboard_df.iloc[0]["params.num_stacks"],
    "num_blocks_per_stack": nbeats_calendar_leaderboard_df.iloc[0]["params.num_blocks_per_stack"],
    "hidden_units": nbeats_calendar_leaderboard_df.iloc[0]["params.hidden_units"],
    "num_layers": nbeats_calendar_leaderboard_df.iloc[0]["params.num_layers"],
    "lr": float(nbeats_calendar_leaderboard_df.iloc[0]["params.lr"]),
    "weight_decay": float(nbeats_calendar_leaderboard_df.iloc[0]["params.weight_decay"]),
    "loss_name": nbeats_calendar_leaderboard_df.iloc[0]["params.loss_name"],
}
BEST_NBEATS_FINAL_CONFIG["final_epochs"] = 100
BEST_NBEATS_FINAL_CONFIG["final_patience"] = 20

print(BEST_NBEATS_FINAL_CONFIG)

{'num_stacks': '5', 'num_blocks_per_stack': '4', 'hidden_units': '512', 'num_layers': '4', 'lr': 0.0001, 'weight_decay': 0.0001, 'loss_name': 'huber', 'final_epochs': 100, 'final_patience': 20}


In [91]:
import mlflow
print(mlflow.get_tracking_uri())
print(mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME))

https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow
<Experiment: artifact_location='mlflow-artifacts:/f940c62366da4ebdbf3b90ec3315ce23', creation_time=1784918403918, effective_trace_archival_retention=None, experiment_id='8', last_update_time=1784918403918, lifecycle_stage='active', name='NBEATS_Training', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>


In [92]:
from kaggle_secrets import UserSecretsClient

os.environ["MLFLOW_TRACKING_USERNAME"] = "LukaBatilashvili07"
os.environ["MLFLOW_TRACKING_PASSWORD"] = UserSecretsClient().get_secret("DAGSHUB_TOKEN")

import dagshub
dagshub.init(repo_owner='LukaBatilashvili07', repo_name='walmart-sales-forecasting', mlflow=True)

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

Initialized MLflow to track repo "LukaBatilashvili07/walmart-sales-forecasting"

Repository LukaBatilashvili07/walmart-sales-forecasting initialized!

<Experiment: artifact_location='mlflow-artifacts:/f940c62366da4ebdbf3b90ec3315ce23', creation_time=1784918403918, effective_trace_archival_retention=None, experiment_id='8', last_update_time=1784918403918, lifecycle_stage='active', name='NBEATS_Training', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [93]:
set_seed(SEED)

final_model_kwargs = dict(
    num_stacks=int(BEST_NBEATS_FINAL_CONFIG["num_stacks"]),
    num_blocks_per_stack=int(BEST_NBEATS_FINAL_CONFIG["num_blocks_per_stack"]),
    hidden_units=int(BEST_NBEATS_FINAL_CONFIG["hidden_units"]),
    num_layers=int(BEST_NBEATS_FINAL_CONFIG["num_layers"]),
)

run_name = (
    f"NBEATS_CalendarAligned_Final_input{CONTEXT_LENGTH}_output{PREDICTION_LENGTH}_"
    f"stacks{final_model_kwargs['num_stacks']}_blocks{final_model_kwargs['num_blocks_per_stack']}_"
    f"hidden{final_model_kwargs['hidden_units']}_layers{final_model_kwargs['num_layers']}_"
    f"lr{BEST_NBEATS_FINAL_CONFIG['lr']:.0e}_wd{BEST_NBEATS_FINAL_CONFIG['weight_decay']:.0e}_"
    f"{BEST_NBEATS_FINAL_CONFIG['loss_name']}"
)

final_model = NBeats(context_length=CONTEXT_LENGTH, prediction_length=PREDICTION_LENGTH, **final_model_kwargs).to(DEVICE)
optimizer = torch.optim.Adam(final_model.parameters(), lr=BEST_NBEATS_FINAL_CONFIG["lr"], weight_decay=BEST_NBEATS_FINAL_CONFIG["weight_decay"])
loss_fn = make_loss_fn(BEST_NBEATS_FINAL_CONFIG["loss_name"])

with mlflow.start_run(run_name=run_name):
    mlflow.log_params({**final_model_kwargs, **BEST_NBEATS_FINAL_CONFIG})

    result = fit_model(
        model=final_model,
        train_loader=nbeats_calendar_data["train_loader"],
        valid_loader=nbeats_calendar_data["valid_loader"],
        optimizer=optimizer,
        loss_fn=loss_fn,
        device=DEVICE,
        epochs=BEST_NBEATS_FINAL_CONFIG["final_epochs"],
        metric_prefix="final",
    )

    final_val_wmae = result["best_valid_wmae"]
    mlflow.log_metric("final_valid_wmae", final_val_wmae)
    mlflow.log_param("run_name_convention", run_name)

print("Final validation WMAE:", final_val_wmae)

Epoch 001 | train_loss=0.26261 | valid_wmae=3493.61 | valid_mae=3027.63
Epoch 002 | train_loss=0.19429 | valid_wmae=3455.29 | valid_mae=3066.32
Epoch 003 | train_loss=0.16391 | valid_wmae=3432.31 | valid_mae=3039.47
Epoch 004 | train_loss=0.14963 | valid_wmae=3435.97 | valid_mae=3045.20
Epoch 005 | train_loss=0.14118 | valid_wmae=3434.93 | valid_mae=3043.16
Epoch 006 | train_loss=0.13552 | valid_wmae=3440.80 | valid_mae=3036.23
Epoch 007 | train_loss=0.13142 | valid_wmae=3446.32 | valid_mae=3046.75
Epoch 008 | train_loss=0.12835 | valid_wmae=3450.68 | valid_mae=3050.84
Epoch 009 | train_loss=0.12479 | valid_wmae=3429.57 | valid_mae=3035.48
Epoch 010 | train_loss=0.12268 | valid_wmae=3459.26 | valid_mae=3064.53
Epoch 011 | train_loss=0.12042 | valid_wmae=3483.41 | valid_mae=3075.48
Epoch 012 | train_loss=0.11844 | valid_wmae=3477.62 | valid_mae=3067.79
Epoch 013 | train_loss=0.11651 | valid_wmae=3504.58 | valid_mae=3084.86
Epoch 014 | train_loss=0.11462 | valid_wmae=3484.64 | valid_mae=

## Save checkpoint + pipeline registry

In [94]:
import mlflow.pyfunc

os.makedirs("checkpoints", exist_ok=True)
checkpoint_path = "checkpoints/nbeats_final.pt"

torch.save(
    {
        "model_state_dict": final_model.state_dict(),
        "model_kwargs": final_model_kwargs,
        "context_length": CONTEXT_LENGTH,
        "prediction_length": PREDICTION_LENGTH,
    },
    checkpoint_path,
)
print("Saved checkpoint to", checkpoint_path)


class NBeatsPipeline(mlflow.pyfunc.PythonModel):
    def __init__(self, base_preprocessor, neural_preprocessor, model,
                 history_panel, cols, context_length, prediction_length, device="cpu"):
        self.base_preprocessor = base_preprocessor
        self.neural_preprocessor = neural_preprocessor
        self.model = model
        self.history_panel = history_panel
        self.cols = cols
        self.context_length = context_length
        self.prediction_length = prediction_length
        self.device = device

    def predict(self, context, model_input: pd.DataFrame) -> pd.DataFrame:
        model_input = model_input.copy()
        model_input["Date"] = pd.to_datetime(model_input["Date"])

        future_base = self.base_preprocessor.transform(model_input)
        future_panel = self.neural_preprocessor.transform(future_base)

        forecast_dataset = WalmartPrecomputedForecastWindowDataset(
            history_df=self.history_panel,
            future_df=future_panel,
            context_length=self.context_length,
            prediction_length=self.prediction_length,
            **self.cols,
        )
        forecast_loader = FastTensorDataLoader(
            forecast_dataset.tensors, batch_size=256, shuffle=False
        )

        preds_matrix = predict_nbeats(self.model, forecast_loader, self.device)

        future_index_df = forecast_dataset.get_future_index().reset_index(drop=True)
        pred_df = future_index_df.copy()
        pred_df["Weekly_Sales"] = np.maximum(preds_matrix.reshape(-1), 0)

        result = model_input[["Store", "Dept", "Date"]].merge(
            pred_df[["Store", "Dept", "Date", "Weekly_Sales"]],
            on=["Store", "Dept", "Date"],
            how="left",
        )
        return result["Weekly_Sales"].to_numpy()


with mlflow.start_run(run_name="NBEATS_Final_Refit_Registry"):
    pipeline_model = NBeatsPipeline(
        base_preprocessor=final_base_preprocessor,
        neural_preprocessor=final_neural_preprocessor,
        model=final_model,
        history_panel=full_train_panel,
        cols=final_nbeats_cols,
        context_length=CONTEXT_LENGTH,
        prediction_length=PREDICTION_LENGTH,
        device=str(DEVICE),
    )
    mlflow.log_metric("final_valid_wmae", final_val_wmae)

    mlflow.pyfunc.log_model(
        artifact_path="nbeats_pipeline",
        python_model=pipeline_model,
        registered_model_name="Walmart_NBeats_Pipeline",
    )

Saved checkpoint to checkpoints/nbeats_final.pt


2026/07/25 16:35:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/25 16:35:29 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
2026/07/25 16:35:29 WARNING mlflow.pyfunc: Failed to infer model signature: Type hint <input: <class 'pandas.core.frame.DataFrame'>, output: <class 'pandas.core.frame.DataFrame'>> cannot be used to infer model signature and input example is not provided, model signature cannot be inferred.
2026/07/25 16:35:45 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local 

🏃 View run NBEATS_Final_Refit_Registry at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/8/runs/f4927de53e31421d97f4d30cba2c424b
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/8
